# Airbnb Paris – Exp 1: AnoLLM
- LoRA-Finetuning Qwen2.5-0.5B auf serialisierten Zeilen, Score = NLL
- Exp 1 = unsupervised: Training auf vollem Train ohne Labels; lesbare Rohwerte + Freitexte

In [ ]:
import sys, time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc
sys.path.insert(0, "../../anollm_src")
from anollm.anollm import AnoLLM

## Daten laden
- Rohdaten einlesen (Aufbereitung folgt in der nächsten Zelle)

In [ ]:
raw = pd.read_csv("../../data/raw/airbnb_paris.csv", low_memory=False)
print("geladen:", raw.shape)

## Spalten droppen & aufbereiten
- ICC-Filter + Label; Leakage-/ID-/Meta-Spalten droppen; lesbare Werte (%-Raten, Counts), NaN behandeln, Split

In [ ]:
text_cols = ["name", "description", "neighborhood_overview", "host_about"]

raw = raw.dropna(subset=["review_scores_rating"])
raw = raw[(raw["review_scores_rating"] == 5.0) | (raw["review_scores_rating"] <= 3.0)].copy()
raw["row_id"] = raw["id"]                                       # stabiler Join-Key (= cleaned row_id)
lab = pd.Series((raw["review_scores_rating"] != 5.0).astype(int).values, index=raw["row_id"].values)  # Outlier=1

# Leakage (review_scores/-counts/-daten) + IDs/URLs/Meta/100%-NaN/Redundanzen entfernen
drop = ["id", "listing_url", "scrape_id", "last_scraped", "source", "picture_url", "host_id", "host_url",
        "host_name", "host_thumbnail_url", "host_picture_url", "license",
        "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin",
        "review_scores_communication", "review_scores_location", "review_scores_value",
        "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly",
        "first_review", "last_review", "reviews_per_month", "calendar_updated", "calendar_last_scraped",
        "host_listings_count", "host_total_listings_count", "minimum_minimum_nights", "maximum_minimum_nights",
        "minimum_maximum_nights", "maximum_maximum_nights", "minimum_nights_avg_ntm", "maximum_nights_avg_ntm",
        "has_availability", "host_neighbourhood", "neighbourhood", "price", "beds", "bathrooms",
        "estimated_revenue_l365d", "neighbourhood_group_cleansed"]
raw = raw.drop(columns=[c for c in drop if c in raw.columns])

# lesbare Werte: %-Raten als Zahl, Listen als Count; Rest roh (Datum/Ort/Kategorien als Strings)
raw["host_response_rate"] = pd.to_numeric(raw["host_response_rate"].astype(str).str.rstrip("%"), errors="coerce")
raw["host_acceptance_rate"] = pd.to_numeric(raw["host_acceptance_rate"].astype(str).str.rstrip("%"), errors="coerce")
raw["amenities_count"] = raw["amenities"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
raw["host_verifications_count"] = raw["host_verifications"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
feat = raw.drop(columns=["amenities", "host_verifications"]).set_index("row_id")

num_cols = [c for c in feat.columns if pd.api.types.is_numeric_dtype(feat[c])]
str_cols = [c for c in feat.columns if c not in num_cols + text_cols]
feat[str_cols] = feat[str_cols].fillna("missing")
feat[text_cols] = feat[text_cols].fillna("")
feat[num_cols] = feat[num_cols].fillna(feat[num_cols].median())

# kanonischer Split (identisch zu baselines/tabpfn) über cleaned-row_id
clean = pd.read_csv("../../data/preprocessed/cleaned_airbnb_paris.csv")
y_clean = (1 - clean["is_top_rating"]).values
tr, te = train_test_split(np.arange(len(clean)), test_size=0.3, stratify=y_clean, random_state=42)
train_ids, test_ids = clean["row_id"].values[tr], clean["row_id"].values[te]
df_train = feat.loc[train_ids].reset_index(drop=True)
df_test = feat.loc[test_ids].reset_index(drop=True)
y_test = lab.loc[test_ids].values
print("train", df_train.shape, "test", df_test.shape, "test outlier rate", round(y_test.mean(), 4))

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_1")

## AnoLLM trainieren & Scores (NLL)
- `max_length_dict` begrenzt Textspalten (Token-Budget); Score = NLL in nativer Orientierung (höher = anomaler)
- Gradient Checkpointing: senkt Trainingsspeicher (Airbnb hat längere Zeilen), Modell bleibt identisch

## Single-GPU-Setup (kein DDP/NCCL)
- Verteilte Env-Variablen entfernen → HF Trainer wrappt nicht in DistributedDataParallel

In [3]:
import os
import torch.distributed as dist
import anollm.anollm_trainer
from torch.utils.data import DataLoader, SequentialSampler

# 1. Alle Cluster-Variablen restlos löschen
for key in ["LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(key, None)

# 2. Falls noch eine alte Prozessgruppe aktiv ist, sauber beenden
if dist.is_available() and dist.is_initialized():
    dist.destroy_process_group()

# 3. MONKEY-PATCH: Wir definieren einen sauberen Single-GPU Dataloader
def single_gpu_get_train_dataloader(self):
    return DataLoader(
        self.train_dataset,
        batch_size=self._train_batch_size,
        sampler=SequentialSampler(self.train_dataset),
        collate_fn=self.data_collator,
        drop_last=True
    )

# 4. Die fehlerhafte Funktion der Amazon-Bibliothek im RAM überschreiben
anollm.anollm_trainer.AnoLLMTrainer.get_train_dataloader = single_gpu_get_train_dataloader

print("🚀 Single-GPU-Modus aktiv, Trainer gepatcht!")

🚀 Single-GPU-Modus aktiv, Trainer gepatcht!


In [ ]:
max_len = {c: 64 for c in text_cols}
model = AnoLLM(llm="Qwen/Qwen2.5-0.5B", efficient_finetuning="lora", textual_columns=text_cols,
               max_length_dict=max_len, batch_size=2, max_steps=2000, learning_rate=5e-4,
               gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
model.model.enable_input_require_grads()  # nötig für Gradient Checkpointing mit LoRA/PEFT

t0 = time.perf_counter()
model.fit(df_train)
scores = model.decision_function(df_test, n_permutations=8, batch_size=2, device="cuda").mean(axis=1)
runtime = time.perf_counter() - t0

scores = np.asarray(scores).astype(float)
auroc = roc_auc_score(y_test, scores)
ap = average_precision_score(y_test, scores)
prec, rec, _ = precision_recall_curve(y_test, scores)
auprc = auc(rec, prec)

with mlflow.start_run(run_name="anollm"):
    mlflow.log_params({"llm": "Qwen/Qwen2.5-0.5B", "max_steps": 2000, "n_permutations": 8})
    mlflow.log_metric("average_precision", ap)
    mlflow.log_metric("auprc", auprc)
    mlflow.log_metric("auc_roc", auroc)
    mlflow.log_metric("runtime_s", runtime)
print(f"anollm: AP={ap:.4f} AUPRC={auprc:.4f} AUC={auroc:.4f} time={runtime:.1f}s")